# Clean `emr_measurement_group`

A first-pass cleaning of the `emr_measurement_group` table (20,826 rows, 5 columns) before analysis.

What this notebook does, in order:
1. **Profile** every column (how full it is, how many distinct values, its type)
2. **Count** duplicates and map id's to measurements that occur multiple times.
3. **Review** columns that hold only a single value
4. **Parse** the date columns (currently stored as text)
5. **Tidy** column types
6. **Save** a cleaned copy + keep an audit note of what changed

The same `profile()` function works on the other `emr_*` tables too, so you can reuse this pattern.

> **Before running:** finish the venv setup in this repo and install the packages — open a terminal and run `pip install pandas pyarrow`. The `requirements.txt` alongside this notebook lists everything.
>
> **Important:** keep your raw patient data in a `data/` folder that is git-ignored. Don't commit EMR exports to GitHub, even a private repo.

## Setup

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

In [7]:
# --- point this at your raw export ---
PATH = Path("../data/emr_measurement_group.csv")

In [8]:
# loading data
df = pd.read_csv(PATH)
df.head()

,id,addedon,addedby,system,code,name,notesection,grouptype,inp,hist,...,histsort,highlight,obflow_from,obflow_to,patientlab,patientinstruction,deletedon,deletedby,ord,showportal
0,1,2010-03-15 12:36:00.809092-07,NaN,True,vitals,Today's Vitals,Vitals,vitals,vitals,htable,...,desc,False,NaN,NaN,0,NaN,NaN,NaN,NaN,0
1,2,2010-03-15 12:36:21.112455-07,NaN,True,recvit,Recent Vitals,NaN,vitals,vitals,htable,...,asc,False,NaN,NaN,0,NaN,NaN,NaN,NaN,0
2,24,2015-11-13 13:27:57.477955-08,1.0,False,manuallab,"Pregnancy Test, Urine",NaN,vitals,vitals,htable,...,desc,False,NaN,NaN,0,NaN,NaN,NaN,NaN,0
3,26,2016-01-18 11:19:40.17852-08,1.0,False,manuallab,Pregnancy Test,NaN,vitals,vitals,htable,...,desc,False,NaN,NaN,0,NaN,NaN,NaN,NaN,0
4,27,2016-01-19 09:19:51.064318-08,1.0,False,manuallab,Progesterone,NaN,vitals,vitals,htable,...,desc,False,NaN,NaN,0,NaN,NaN,NaN,NaN,0


## 1. Profile every column

One row per column so you can see at a glance what's worth keeping.

In [9]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: how full it is, how many distinct values, and its dtype."""
    out = pd.DataFrame({
        "non_null":  frame.notna().sum(),
        "nulls":     frame.isna().sum(),
        "distinct":  frame.nunique(dropna=True),
        "dtype":     frame.dtypes.astype(str),
    })
    out["pct_null"] = (out["nulls"] / len(frame) * 100).round(1)
    return out.sort_values("non_null", ascending=False)

In [11]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: how full it is, how many distinct values, its dtype,
    and whether all non-null values are identical."""
    out = pd.DataFrame({
        "non_null": frame.notna().sum(),
        "nulls": frame.isna().sum(),
        "distinct": frame.nunique(dropna=True),
        "dtype": frame.dtypes.astype(str),
    })

    out["pct_null"] = (out["nulls"] / len(frame) * 100).round(1)

    # True if all non-null values in the column are the same
    out["all_values_same"] = out["distinct"] <= 1

    return out.sort_values("non_null", ascending=False)

In [12]:
prof = profile(df)
prof

,non_null,nulls,distinct,dtype,pct_null,all_values_same
id,69,0,69,int64,0.0,False
addedon,69,0,65,str,0.0,False
system,69,0,2,bool,0.0,False
code,69,0,34,str,0.0,False
name,69,0,57,str,0.0,False
grouptype,69,0,9,str,0.0,False
printone,69,0,3,str,0.0,False
printhist,69,0,1,str,0.0,True
hist,69,0,1,str,0.0,True
patientlab,69,0,1,int64,0.0,True


In [13]:
# dropping columns with all the same value
cols_to_drop = prof.index[
    (prof["non_null"] == 0) | (prof["all_values_same"])
].tolist()

print(f"Dropping {len(cols_to_drop)} empty or constant columns:\n")
for c in cols_to_drop:
    print("  -", c)

df = df.drop(columns=cols_to_drop)

print(f"\nRemaining: {df.shape[1]} columns")


Dropping 11 empty or constant columns:

  - printhist
  - hist
  - patientlab
  - agefrom
  - ageto
  - inp
  - obflow_from
  - patientinstruction
  - obflow_to
  - deletedby
  - deletedon

Remaining: 14 columns


In [ ]:
# list all distinct values of label column
print(df["name"].unique())

# now a count of each label
print(df["name"].value_counts())

# hCG                       3
# FSH                       3
# Progesterone              2
# LH                        2
# RPR                       2
# E2                        2
# AMH                       2
# Semen Culture             2
# Blood Tests               2
# Screening Tests           2

<StringArray>
[                                                         'Today's Vitals',
                                                           'Recent Vitals',
                                                   'Pregnancy Test, Urine',
                                                          'Pregnancy Test',
                                                            'Progesterone',
                                                            'Estradiol E2',
                                                                     'hCG',
                                                                      'LH',
                                                                     'FSH',
                                                                     'RPR',
                                                                     'CBC',
                                                              'Blood type',
                                                            'OB Flowsheet'

In [23]:
import pandas as pd
import numpy as np

patterns = {
    r"Progesterone": "Progesterone",
    r"hCG": "hCG",
    r"FSH": "FSH",
    r"\bLH\b": "LH",
    r"RPR": "RPR",
    r"\bE2\b": "E2",
    r"AMH": "AMH",
    r"Semen Culture": "Semen Culture",
    r"Blood Tests": "Blood Tests",
    r"Screening Tests": "Screening Tests",
}

df["category_name"] = df["name"]

for pattern, category in patterns.items():
    mask = df["name"].str.contains(pattern, case=False, na=False, regex=True)
    df.loc[mask, "category_name"] = category

# hCG                       3
# FSH                       3
# Progesterone              2
# LH                        2
# RPR                       2
# E2                        2
# AMH                       2
# Semen Culture             2
# Blood Tests               2
# Screening Tests           2

In [24]:
display(df.head(30))

,id,addedon,addedby,system,code,name,notesection,grouptype,printone,histcount,histsort,highlight,ord,showportal,category_name
0,1,2010-03-15 12:36:00.809092-07,NaN,True,vitals,Today's Vitals,Vitals,vitals,hlist,5,desc,False,NaN,0,Today's Vitals
1,2,2010-03-15 12:36:21.112455-07,NaN,True,recvit,Recent Vitals,NaN,vitals,hlist,5,asc,False,NaN,0,Recent Vitals
2,24,2015-11-13 13:27:57.477955-08,1.0,False,manuallab,"Pregnancy Test, Urine",NaN,vitals,vlist,5,desc,False,NaN,0,"Pregnancy Test, Urine"
3,26,2016-01-18 11:19:40.17852-08,1.0,False,manuallab,Pregnancy Test,NaN,vitals,vlist,5,desc,False,NaN,0,Pregnancy Test
4,27,2016-01-19 09:19:51.064318-08,1.0,False,manuallab,Progesterone,NaN,vitals,vlist,5,desc,False,NaN,0,Progesterone
5,28,2016-01-19 09:19:51.064318-08,1.0,False,manuallab,Estradiol E2,NaN,vitals,vlist,5,desc,False,NaN,0,E2
6,29,2016-01-19 09:19:51.064318-08,1.0,False,manuallab,hCG,NaN,vitals,vlist,5,desc,False,NaN,0,hCG
7,30,2016-01-19 09:19:51.064318-08,1.0,False,manuallab,LH,NaN,vitals,vlist,5,desc,False,NaN,0,LH
8,31,2016-01-19 09:19:51.064318-08,1.0,False,manuallab,FSH,NaN,vitals,vlist,5,desc,False,NaN,0,FSH
9,37,2016-08-18 13:45:09.897668-07,1.0,False,manuallab,RPR,NaN,vitals,vlist,5,desc,False,NaN,0,RPR
